In [2]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

C:\Users\ankit\AppData\Local\Temp\ipykernel_37772\1007429996.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


CUDA available: True


In [5]:
#importing dataset



# 1. Initialize the loader with your PDF file
loader = PyPDFLoader("S:/Projects/rag-dissertation/datasets/Enterprise Rag/957f31e694198a1695d14aa512a972ab6f3c55b5.pdf")

# 2. Load and split the PDF into a list of Documents (one per page)
docs = loader.load()

# Or load and split into chunks directly
pages = loader.load_and_split()

print(f"Total pages loaded: {len(docs)}")
print(docs[0].page_content)  # Content of the first page
print(docs[0].metadata)  # Page number and source path

Total pages loaded: 152
2022  
Annual Report
{'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-03-03T18:25:35-05:00', 'author': '', 'moddate': '2023-03-08T08:55:07-05:00', 'title': '385703_002_BMK_WEB.PDF', 'source': 'S:/Projects/rag-dissertation/datasets/Enterprise Rag/957f31e694198a1695d14aa512a972ab6f3c55b5.pdf', 'total_pages': 152, 'page': 0, 'page_label': '1'}


In [ ]:
#loading dataset in langchain

# docs = []
# for col in df.columns:
#     if col == "Unnamed: 0":
#         continue
#     text = "\n".join(df[col].dropna().astype(str))
#     docs.append(Document(page_content=text, metadata={"column": col}))

# print(len(docs))   # = number of columns
# print(docs[0].metadata)
# print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [6]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

779
reducing weight and mass while also improving range 
and enabling faster charging — critical factors for 
BEV adoption. We have a comprehensive high-voltage 
portfolio, which we expanded during 2022 to include 
integrated power electronics. This solution combines six 
devices into one while also hosting battery management 
system software, and we won our ﬁrst program to 
deliver this integrated solution to a major OEM.
During 2022, Aptiv also acquired an 85 percent 
equity stake in Intercable Automotive Solutions, a 
pioneer in the design and manufacture of high-voltage 
power distribution and interconnect technology. The 
company’s industry-leading busbar technologies 
enhance our high-voltage product portfolio, and 
Intercable Automotive is able to leverage Aptiv’s  
global scale and manufacturing footprint.
THE NEXT CHAPTER FOR APTIV
Consumers increasingly are demanding advanced, 
software-enabled features, which need to be supported


In [7]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 608.33it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [8]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="Enterprise_Embeddings",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


Loaded 0 chunks from S:\Projects\rag-dissertation\src\chroma_langchain_db


In [9]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = True

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


100%|██████████| 1/1 [01:53<00:00, 113.15s/it]


In [10]:
# Retrieve evidence for the same question that will be evaluated.
query = "On what date did Aptiv acquired 85\% of the equity interests of Intercable Automotive Solutions??"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [creationdate=2023-03-03T18:25:35-05:00, author=, moddate=2023-03-08T08:55:07-05:00, page=77, producer=Acrobat Distiller 8.1.0 (Windows), creator=Adobe InDesign 18.1 (Macintosh), title=385703_002_BMK_WEB.PDF, page_label=78, total_pages=152, source=S:/Projects/rag-dissertation/datasets/Enterprise Rag/957f31e694198a1695d14aa512a972ab6f3c55b5.pdf] ---
investment and represented a returt n on investment in cash flff ows frff om operating activities.
Aptiv's equity investments without readily determinabla e faff ir value totaled $67 million and $30 million as of
December 31, 2022 and 2021, respectively, and are classififf ed within other long-term assets in the consolidated balance sheets.
Aptiv's investments in publicly traded equity securities totaled $17 million and $66 million as of December 31, 2022 and 2021,
respectively, and are classififf ed within other long-term assets in the consolidated balance sheet. Refeff r to Note 5. Investments in
A

In [11]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [12]:
# Generate the RAG answer from the retrieved evidence.
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


November 30, 2022


In [ ]:
# Sanity check the local Ollama server with the judge model.
import ollama

r = ollama.chat(
    model="qwen3.5:2b",
    messages=[{"role": "user", "content": 'Say hi in JSON: {"msg": ...}'}],
)
print(repr(r.message.content))


In [13]:
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    "On November 30, 2022, Aptiv acquired 85\% of the equity interests of Intercable Automotive Solutions S.r.l. (“Intercable Automotive."
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    temperature=0,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\ankit\AppData\Local\Temp\ipykernel_37772\4014484381.py:63: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  "hallucination": HallucinationMetric(


hallucination: 0.000
  passed: False
  reason: The hallucination score is 0.00 because the actual output correctly aligns with the provided factual alignments by including all necessary details (company names, acquisition percentage, and date) without any contradictions or missing information.


faithfulness: 0.500
  passed: True
  reason: The faithfulness score is 0.50 because the actual output incorrectly claims the text is dated November 30, 2022, while the retrieval context explicitly states that Aptiv acquired Intercabla on this specific date and lists it as a fact about the acquisition.


answer relevancy: 0.500
  passed: True
  reason: The score is 0.50 because the output incorrectly claims the text contains only a date, which directly contradicts the input question asking for the acquisition date.
